## Popcorn Hack 1: Check One SFI Part Record

**Prediction:**
- Initial record (`category = "Auto Racing"`): All 4 field checks (`has_product_name`, `valid_category`, `has_spec_number`, `has_effective_date`) will evaluate to `True`, so `is_complete_and_valid` will be `True`.
- Modified record (`category = "Street Car"`): `valid_category` will evaluate to `False`, making `is_complete_and_valid` evaluate to `False`.

In [ ]:
# Popcorn Hack 1 - Check SFI Part Fields

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]

def check_part_fields(record):
    has_product_name = record["product_name"] != ""
    valid_category = (record["category"] == "Auto Racing") or (record["category"] == "Drag Racing")
    has_spec_number = record["spec_number"] != ""
    has_effective_date = record["effective_date"] != ""
    is_complete = has_product_name and valid_category and has_spec_number and has_effective_date

    print("Product Name Present:", has_product_name)
    print("Valid Racing Category:", valid_category)
    print("Spec Number Present:", has_spec_number)
    print("Effective Date Present:", has_effective_date)
    print("All Requirements Met:", is_complete)

print("--- Run 1: Original SFI Record ---")
check_part_fields(part)

# Change one input to test the opposite case
part_modified = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Street Car",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

print("\n--- Run 2: Modified Category ('Street Car') ---")
check_part_fields(part_modified)

--- Run 1: Original SFI Record ---
Product Name Present: True
Valid Racing Category: True
Spec Number Present: True
Effective Date Present: True
All Requirements Met: True

--- Run 2: Modified Category ('Street Car') ---
Product Name Present: True
Valid Racing Category: False
Spec Number Present: True
Effective Date Present: True
All Requirements Met: False


## Popcorn Hack 2: Backend Create Rule

**Prediction:**
- Run 1 (Starter record with `spec_number = "1.1"`): Even though all required fields are present and the category is valid, `"1.1"` is already in `existing_spec_numbers` (`["1.1", "2.1"]`), so `duplicate_spec` is `True`, `not duplicate_spec` is `False`, and the record is **REJECTED**.
- Run 2 (Modified record with `spec_number = "1.5"`): All fields are valid and `"1.5"` is not a duplicate, so `can_create_record` is `True` and the record is **ACCEPTED**.

In [ ]:
# Popcorn Hack 2 - Backend Create Rule

part = {
    "product_name": "Replacement Flywheels and Clutch Assemblies",
    "category": "Auto Racing",
    "spec_number": "1.1",
    "effective_date": "Nov. 9, 2001"
}

valid_categories = ["Auto Racing", "Drag Racing"]
existing_spec_numbers = ["1.1", "2.1"]

def evaluate_create_rule(record):
    has_product_name = record["product_name"] != ""
    valid_category = record["category"] in valid_categories
    has_spec_number = record["spec_number"] != ""
    has_effective_date = record["effective_date"] != ""
    duplicate_spec = (record["spec_number"] == "1.1") or (record["spec_number"] == "2.1")

    can_create_record = (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and not duplicate_spec
    )

    print(f"Testing Spec {record['spec_number']} ({record['product_name']}):")
    print("  Duplicate Spec:", duplicate_spec)
    print("  Can Create Record:", can_create_record)
    if can_create_record:
        print("  Decision: ACCEPT RECORD (Create in Database)")
    else:
        print("  Decision: REJECT RECORD (Do Not Create)")

print("--- Case 1: Starter Record (Duplicate Spec 1.1) ---")
evaluate_create_rule(part)

# Valid non-duplicate test case
new_valid_part = {
    "product_name": "Bellhousing Safety Shield",
    "category": "Drag Racing",
    "spec_number": "1.5",
    "effective_date": "Sep. 15, 2026"
}

print("\n--- Case 2: New Valid Record (Spec 1.5) ---")
evaluate_create_rule(new_valid_part)

--- Case 1: Starter Record (Duplicate Spec 1.1) ---
Testing Spec 1.1 (Replacement Flywheels and Clutch Assemblies):
  Duplicate Spec: True
  Can Create Record: False
  Decision: REJECT RECORD (Do Not Create)

--- Case 2: New Valid Record (Spec 1.5) ---
Testing Spec 1.5 (Bellhousing Safety Shield):
  Duplicate Spec: False
  Can Create Record: True
  Decision: ACCEPT RECORD (Create in Database)


## Popcorn Hack 3: SFI Search Filter

**Prediction:**
- Query 1 (`query = "1.2"`): Will match only `"Multiple Disc Clutch Assemblies"` because its `spec_number == "1.2"` and its category `"Drag Racing"` is in `valid_categories`. The other records do not match `"1.2"`.
- Query 2 (`query = "Flywheel"`): Will match `"Replacement Flywheels and Clutch Assemblies"` (`1.1`) and `"Racing Flywheel Record"` (`2.1`) because `"Flywheel"` is in their `product_name` AND both have valid racing categories (`"Auto Racing"`). If a Street Car flywheel record is added, it will be filtered out because its category is not accepted.

In [ ]:
# Popcorn Hack 3 - SFI Search Filter

query = "1.2"

records = [
    {
        "product_name": "Replacement Flywheels and Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.1"
    },
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Drag Racing",
        "spec_number": "1.2"
    },
    {
        "product_name": "Racing Flywheel Record",
        "category": "Auto Racing",
        "spec_number": "2.1"
    },
    {
        "product_name": "Street Flywheel Kit",
        "category": "Street Car",
        "spec_number": "3.1"
    }
]

valid_categories = ["Auto Racing", "Drag Racing"]

def search_sfi_records(search_query, record_list):
    print(f"=== Search Query: '{search_query}' ===")
    matches_found = 0
    for rec in record_list:
        matches_name = search_query.lower() in rec["product_name"].lower()
        matches_spec = rec["spec_number"] == search_query
        matches_query = matches_name or matches_spec
        accepted_category = (rec["category"] == "Auto Racing") or (rec["category"] == "Drag Racing")

        is_valid_match = matches_query and accepted_category

        if is_valid_match:
            matches_found += 1
            print(f"  MATCH: {rec['product_name']} | Category: {rec['category']} | Spec: {rec['spec_number']}")
        else:
            print(f"  SKIP:  {rec['product_name']} (matches_query={matches_query}, accepted_category={accepted_category})")
    print(f"Total Valid Matches: {matches_found}\n")

# Run 1 with default query "1.2"
search_sfi_records(query, records)

# Run 2 with changed query "Flywheel"
search_sfi_records("Flywheel", records)

=== Search Query: '1.2' ===
  SKIP:  Replacement Flywheels and Clutch Assemblies (matches_query=False, accepted_category=True)
  MATCH: Multiple Disc Clutch Assemblies | Category: Drag Racing | Spec: 1.2
  SKIP:  Racing Flywheel Record (matches_query=False, accepted_category=True)
  SKIP:  Street Flywheel Kit (matches_query=False, accepted_category=False)
Total Valid Matches: 1

=== Search Query: 'Flywheel' ===
  MATCH: Replacement Flywheels and Clutch Assemblies | Category: Auto Racing | Spec: 1.1
  SKIP:  Multiple Disc Clutch Assemblies (matches_query=False, accepted_category=True)
  MATCH: Racing Flywheel Record | Category: Auto Racing | Spec: 2.1
  SKIP:  Street Flywheel Kit (matches_query=True, accepted_category=False)
Total Valid Matches: 2



## In-Class MCQ Check & College Board Pseudocode Comparison

### MCQ Result Line
`MCQ 3.05: 4/4 | answers: A,C,A,B`

### Section 3 Pseudocode Check
Tracing the first homework test record (`product_name = "Multiple Disc Clutch Assemblies"`, `category = "Auto Racing"`, `spec_number = "1.2"`, `effective_date = "Feb. 9, 2006"`) against the College Board pseudocode:
1. `hasProductName ← (productName ≠ "")` evaluates to `true`.
2. `validCategory ← (category = "Auto Racing") OR (category = "Drag Racing")` evaluates to `true OR false` → `true`.
3. `hasSpecNumber ← (specNumber ≠ "")` evaluates to `true`.
4. `hasEffectiveDate ← (effectiveDate ≠ "")` evaluates to `true`.
5. `duplicateSpec ← (specNumber = "1.1") OR (specNumber = "2.1")` evaluates to `false OR false` → `false`.
6. `isValid ← hasProductName AND validCategory AND hasSpecNumber AND hasEffectiveDate AND NOT duplicateSpec` evaluates to `true AND true AND true AND true AND true` → `true`.
7. Both the College Board pseudocode and the Python validator output `ACCEPT RECORD`.

## Homework Hack (Independent): SFI Backend Validator

**Boolean Logic Explanation:**
- Each candidate record is checked across five individual Boolean conditions:
  1. `has_product_name`: checks `record["product_name"].strip() != ""`
  2. `valid_category`: uses `or` (`record["category"] == "Auto Racing" or record["category"] == "Drag Racing"`) so either approved racing category is accepted.
  3. `has_spec_number`: checks `record["spec_number"].strip() != ""`
  4. `has_effective_date`: checks `record["effective_date"].strip() != ""`
  5. `duplicate_spec`: checks whether `record["spec_number"] in existing_spec_numbers` (`"1.1"` or `"2.1"`), and applies `not duplicate_spec` so existing spec numbers are rejected.
- All five conditions are combined with `and` into `is_valid` because every requirement must pass before the backend accepts a candidate SFI record.

In [ ]:
# Homework Hack - SFI Backend Validator

existing_spec_numbers = ["1.1", "2.1"]
valid_categories = ["Auto Racing", "Drag Racing"]

test_records = [
    {
        "product_name": "Multiple Disc Clutch Assemblies",
        "category": "Auto Racing",
        "spec_number": "1.2",
        "effective_date": "Feb. 9, 2006"
    },
    {
        "product_name": "Replacement Flywheels",
        "category": "Street Car",
        "spec_number": "3.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "Existing Flywheel Record",
        "category": "Drag Racing",
        "spec_number": "1.1",
        "effective_date": "Jan. 1, 2026"
    },
    {
        "product_name": "",
        "category": "Drag Racing",
        "spec_number": "4.2",
        "effective_date": "Mar. 10, 2026"
    }
]

def validate_sfi_record(record, existing_specs, accepted_categories):
    has_product_name = record["product_name"].strip() != ""
    valid_category = (
        record["category"] == accepted_categories[0]
        or record["category"] == accepted_categories[1]
    )
    has_spec_number = record["spec_number"].strip() != ""
    has_effective_date = record["effective_date"].strip() != ""
    duplicate_spec = record["spec_number"] in existing_specs

    is_valid = (
        has_product_name
        and valid_category
        and has_spec_number
        and has_effective_date
        and not duplicate_spec
    )

    return is_valid, {
        "has_product_name": has_product_name,
        "valid_category": valid_category,
        "has_spec_number": has_spec_number,
        "has_effective_date": has_effective_date,
        "not_duplicate_spec": not duplicate_spec
    }

for i, candidate in enumerate(test_records, start=1):
    accepted, checks = validate_sfi_record(candidate, existing_spec_numbers, valid_categories)
    status = "ACCEPT RECORD" if accepted else "REJECT RECORD"
    name_display = candidate["product_name"] if candidate["product_name"] else "<Missing Name>"
    print(f"Record {i}: {name_display} (Spec: {candidate['spec_number']}, Category: {candidate['category']})")
    print(f"  Checks -> {checks}")
    print(f"  Result -> {status}\n")

Record 1: Multiple Disc Clutch Assemblies (Spec: 1.2, Category: Auto Racing)
  Checks -> {'has_product_name': True, 'valid_category': True, 'has_spec_number': True, 'has_effective_date': True, 'not_duplicate_spec': True}
  Result -> ACCEPT RECORD

Record 2: Replacement Flywheels (Spec: 3.1, Category: Street Car)
  Checks -> {'has_product_name': True, 'valid_category': False, 'has_spec_number': True, 'has_effective_date': True, 'not_duplicate_spec': True}
  Result -> REJECT RECORD

Record 3: Existing Flywheel Record (Spec: 1.1, Category: Drag Racing)
  Checks -> {'has_product_name': True, 'valid_category': True, 'has_spec_number': True, 'has_effective_date': True, 'not_duplicate_spec': False}
  Result -> REJECT RECORD

Record 4: <Missing Name> (Spec: 4.2, Category: Drag Racing)
  Checks -> {'has_product_name': False, 'valid_category': True, 'has_spec_number': True, 'has_effective_date': True, 'not_duplicate_spec': True}
  Result -> REJECT RECORD

